# Mini-GPT on Google Colab (GPU)

Train the decoder-only Transformer from the **Mini-GPT** repo on a free Colab GPU.

You can train on **either** dataset:
- **Shakespeare** (default) -> run sections 1-4, then 6-8.
- **Hinglish** (Hindi-English, Roman script) -> run sections 1-3, then **5**, then 6-8.

**Before running:** set the runtime to GPU via **Runtime -> Change runtime type -> Hardware accelerator: GPU**.

## 1. Check the GPU
Confirm a GPU is attached. If this errors or shows nothing, set the runtime to GPU first.

In [ ]:
!nvidia-smi

## 2. Clone the repo
Pulls the exact code (data pipeline, attention, blocks, model, training loop) from GitHub.

In [ ]:
!git clone https://github.com/shubham-KMX/Mini-GPT.git
%cd Mini-GPT

## 3. Verify PyTorch sees CUDA
Colab ships with PyTorch preinstalled, so no `pip install` is normally needed.

In [ ]:
import torch
print('torch', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 4. (Shakespeare mode) Use the default dataset
Skip this section if you want Hinglish. Otherwise there is nothing to do here -- `train.py` downloads Tiny Shakespeare to `input.txt` automatically the first time it runs.

## 5. (Hinglish mode) Build a Hinglish corpus
Run **only** if you want the model to learn Hinglish instead of Shakespeare.

`prepare_hinglish.py` streams a Hinglish (Hindi-English, Roman script) corpus, cleans it (removes URLs / @mentions / # symbols), and writes ~5 MB to `input.txt` -- overwriting Shakespeare so the training step below needs no extra flags.

> If the default dataset id fails to load, pass a working one with `--dataset <hf_dataset_id>`.

In [ ]:
!pip install -q datasets
!python prepare_hinglish.py --target-mb 5
# Peek at the prepared data
print('\n--- First 500 characters of input.txt ---')
print(open('input.txt', encoding='utf-8').read()[:500])

## 6. Train
`train.py` auto-detects CUDA and reads `input.txt` (Shakespeare or Hinglish, whichever you set up above).

This config (256-dim, 6 layers, 6 heads, context 256) trains in a few minutes on a T4 and reaches a validation loss around 1.5.

In [ ]:
!python train.py \
  --batch-size 64 \
  --block-size 256 \
  --n-embd 256 \
  --n-head 6 \
  --n-layer 6 \
  --dropout 0.2 \
  --learning-rate 3e-4 \
  --max-iters 5000 \
  --eval-interval 500 \
  --gen-tokens 1000

## 7. Generate from your own prompt
`train.py` saves `mini_gpt.pt`. Reload it and sample without retraining. Change `prompt` to anything you like.

- Shakespeare model: try `ROMEO:` or `KING:`
- Hinglish model: try `Aaj kal` or `Mujhe lagta hai`

In [ ]:
import torch
from model import GPTLanguageModel

device = 'cuda' if torch.cuda.is_available() else 'cpu'
ckpt = torch.load('mini_gpt.pt', map_location=device)
cfg = ckpt['config']
chars = ckpt['vocab']

# Rebuild the tokenizer mappings from the saved vocabulary
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
decode = lambda ids: ''.join(itos[i] for i in ids)

def encode(s):
    # Skip any character not in the trained vocabulary
    return [stoi[c] for c in s if c in stoi]

model = GPTLanguageModel(
    vocab_size=len(chars),
    n_embd=cfg['n_embd'],
    block_size=cfg['block_size'],
    n_layer=cfg['n_layer'],
    n_head=cfg['n_head'],
    dropout=cfg['dropout'],
).to(device)
model.load_state_dict(ckpt['model_state'])
model.eval()

prompt = 'Aaj kal'  # <-- change this
ids = encode(prompt) or encode('\n')
context = torch.tensor([ids], dtype=torch.long, device=device)
out = model.generate(context, max_new_tokens=1000)
print(decode(out[0].tolist()))

## 8. (Optional) Download the checkpoint
Save the trained weights to your machine so you can run `generate.py` locally.

In [ ]:
from google.colab import files
files.download('mini_gpt.pt')